# Araucaria angustifolia: nearby tree counting with buffers

In [16]:
import geopandas as gpd

## 1. Parameters
Change the buffer size (in meters) and re-run the notebook.

In [17]:
buffer_size = 5_000  # meters

## 2. Load data
Reproject from lat/lon (EPSG:4326) to a metric CRS (EPSG:5880) so the buffer is in meters.

In [18]:
gpkg_path = '/home/jovyan/geo-data-science/GBIF_Araucaria_angustifolia.gpkg'
trees = gpd.read_file(gpkg_path).to_crs(epsg=5880)
trees = trees[["key", "year", "stateProvince", "geometry"]]
print(len(trees))

4648


In [19]:
trees.head(5)

,key,year,stateProvince,geometry
0,5938070397,2026.0,Santa Catarina,POINT (5431299.943 6886309.944)
1,5938345686,2026.0,Santa Catarina,POINT (5396126.236 6863682.078)
2,5938381564,2026.0,Rio Grande do Sul,POINT (5167816.858 6745712.218)
3,5938442452,2026.0,Paraná,POINT (5505139.92 7181554.525)
4,5938507472,2026.0,Paraná,POINT (5098229.079 7322533.377)


In [20]:
esri_tiles = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"
esri_attr = "Tiles &copy; Esri &mdash; Esri, HERE, Garmin, © OpenStreetMap contributors, and the GIS user community"

trees.explore(column="year", cmap="viridis", tiles=esri_tiles, attr=esri_attr, max_zoom=19,
                legend_kwds={"caption": "Araucárias"})

## 3. Nearby counting
Buffer each tree, then count how many trees fall inside each buffer (excluding the tree itself).

In [21]:
buffers = trees.copy()
buffers["geometry"] = trees.buffer(buffer_size)

# Spatial join: one row for each (buffer, tree inside it) pair
pairs = gpd.sjoin(buffers, trees, predicate="contains")

# Count pairs per buffer and subtract 1 (the tree itself)
trees["nearby_counting"] = pairs.groupby(level=0).size() - 1
trees["nearby_counting"].describe()

count    4648.000000
mean       65.053787
std        98.476661
min         0.000000
25%         4.000000
50%        29.000000
75%        75.000000
max       352.000000
Name: nearby_counting, dtype: float64

## 4. Interactive map

In [24]:
trees.sort_values("nearby_counting").explore(
    column="nearby_counting", cmap="viridis", tiles=esri_tiles, attr=esri_attr, max_zoom=19,
    legend_kwds={"caption": f"Trees within {buffer_size} m"})

In [25]:
# Example tree: the one with the most neighbors (any index of `trees` works)
example_id = trees["nearby_counting"].idxmax()
example_tree = trees.loc[[example_id]]
example_buffer = example_tree.buffer(buffer_size)

m = trees.sort_values("nearby_counting").explore(
    column="nearby_counting", cmap="viridis", tiles=esri_tiles, attr=esri_attr, max_zoom=19,
    legend_kwds={"caption": f"Trees within {buffer_size} m"})

# Draw the buffer and the example tree on top of the same map
example_buffer.explore(m=m, color="red", style_kwds={"fillOpacity": 0.1, "weight": 3})
example_tree.explore(m=m, color="red", marker_kwds={"radius": 8},
                     tooltip=["key", "nearby_counting"])

# Zoom to the buffer
xmin, ymin, xmax, ymax = example_buffer.to_crs(epsg=4326).total_bounds
m.fit_bounds([[ymin, xmin], [ymax, xmax]])
m

In [26]:
unique_trees = trees.drop_duplicates(subset="geometry").drop(columns="nearby_counting")
print(f"{len(trees)} records -> {len(unique_trees)} unique locations")

buffers = unique_trees.copy()
buffers["geometry"] = unique_trees.buffer(buffer_size)
pairs = gpd.sjoin(buffers, unique_trees, predicate="contains")
unique_trees["nearby_counting"] = pairs.groupby(level=0).size() - 1
unique_trees["nearby_counting"].describe()

4648 records -> 3453 unique locations


count    3453.000000
mean       33.339125
std        47.090098
min         0.000000
25%         2.000000
50%        15.000000
75%        34.000000
max       251.000000
Name: nearby_counting, dtype: float64

In [27]:
# Same example location as before, now counting unique points only
example_unique = unique_trees[unique_trees.geometry == example_tree.geometry.iloc[0]]
print(f"Example tree: {example_tree['nearby_counting'].iloc[0]} records -> "
      f"{example_unique['nearby_counting'].iloc[0]} unique points within {buffer_size} m")

m = unique_trees.sort_values("nearby_counting").explore(
    column="nearby_counting", cmap="viridis", tiles=esri_tiles, attr=esri_attr, max_zoom=19,
    legend_kwds={"caption": f"Unique points within {buffer_size} m"})
example_buffer.explore(m=m, color="red", style_kwds={"fillOpacity": 0.1, "weight": 3})
example_unique.explore(m=m, color="red", marker_kwds={"radius": 8},
                       tooltip=["key", "nearby_counting"])
m.fit_bounds([[ymin, xmin], [ymax, xmax]])
m

Example tree: 352 records -> 11 unique points within 5000 m
